In [18]:
import cell2mol
from cell2mol.readwrite import readinfo
from cell2mol.tmcharge_common import cell
from cell2mol.c2m_module import reconstruct
from cell2mol.formal_charge import (drive_get_poscharges,
                                    classify_mols,
                                    balance_charge,
                                    build_bonds,
                                    prepare_mols,
                                    prepare_unresolved)

In [2]:
infopath = "INOVAL/INOVAL.info"

In [7]:
refcode = 'INOVAL'

In [6]:
debug = 1

In [5]:
labels, pos, ref_labels, ref_fracs, cellvec, cellparam = readinfo(infopath)

In [10]:
# Initialize cell object
warning_list = []
newcell = cell(refcode, labels, pos, cellvec, cellparam, warning_list)
if debug >= 1: print("[Refcode]", newcell.refcode)

[Refcode] INOVAL


In [13]:
# Cell Reconstruction
if debug >= 1: print("===================================== step 1 : Cell reconstruction =====================================\n")
newcell = reconstruct(newcell, ref_labels, ref_fracs, debug=debug)

===================================== step 1 : Cell reconstruction =====================================

Molecule : H2-O-Cl3-Fe
Metal : Fe	Coordinating atoms : ['Cl', 'Cl', 'Cl', 'O']
Coordination number : 4 {'Tetrahedral': 0.943, 'Square planar': 28.56, 'Seesaw': 6.482}
The most likely geometry : 'Tetrahedral' with deviation value 0.943 (hapticity : False)

Molecule : H2-O-Cl3-Fe
Metal : Fe	Coordinating atoms : ['Cl', 'Cl', 'Cl', 'O']
Coordination number : 4 {'Tetrahedral': 0.943, 'Square planar': 28.56, 'Seesaw': 6.482}
The most likely geometry : 'Tetrahedral' with deviation value 0.943 (hapticity : False)

Molecule : Cl6-Fe2
Metal : Fe	Coordinating atoms : ['Cl', 'Cl', 'Cl', 'Cl']
Coordination number : 4 {'Tetrahedral': 0.745, 'Square planar': 33.032, 'Seesaw': 6.486}
The most likely geometry : 'Tetrahedral' with deviation value 0.745 (hapticity : False)

Molecule : Cl6-Fe2
Metal : Fe	Coordinating atoms : ['Cl', 'Cl', 'Cl', 'Cl']
Coordination number : 4 {'Tetrahedral': 0.745, 'Squa

In [15]:
for ref in newcell.refmoleclist:
    print(ref.formula)

H2-O-Cl3-Fe
H2-O-Cl3-Fe
H10-C4-O
H36-C16-N
Cl6-Fe2
H10-C4-O
H36-C16-N
H36-C16-N
H36-C16-N


In [16]:
if debug >= 1: print("===================================== step 2 : Charge Assignment =======================================\n")

===================================== step 2 : Charge Assignment =======================================



In [20]:
# Indentify unique chemical species
molec_indices, ligand_indices, unique_indices, unique_species = classify_mols(newcell.moleclist, debug=debug)

In [21]:
molec_indices

[0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 2, 3, 4, 4, 4, 4, 4, 5, 6, 7, 8]

In [23]:
unique_indices

[0, 0, 1, 1, 0, 0, 2, 2, 3, 4, 0, 0, 0, 5, 3, 4, 0, 0, 0, 5, 6, 6, 6, 6]

In [24]:
unique_species

[['Ligand',
 ['Ligand',
 ['Metal',
 ['Other', <cell2mol.tmcharge_common.molecule at 0x1c3b4efe0>],
 ['Ligand',
 ['Metal',
 ['Other', <cell2mol.tmcharge_common.molecule at 0x1c3b4fe50>]]

In [25]:
# Group all unique species in a cell variable
for spec in unique_species:            # spec is a list in which item 1 is the actual unique specie
    print(spec[0], spec[1])
    newcell.speclist.append(spec[1])    
if len(unique_species) == 0:
    if debug >= 1: print("Empty list of species found. Stopping")
    sys.exit()
else:
    if debug >= 1: print(f"{len(unique_species)} Species (Ligand or Molecules) to Characterize")

Ligand <cell2mol.tmcharge_common.ligand object at 0x1c3b4e350>
Ligand <cell2mol.tmcharge_common.ligand object at 0x10f1f6530>
Metal <cell2mol.tmcharge_common.metal object at 0x1c3b4e320>
Other <cell2mol.tmcharge_common.molecule object at 0x1c3b4efe0>
Ligand <cell2mol.tmcharge_common.ligand object at 0x1c3b4ecb0>
Metal <cell2mol.tmcharge_common.metal object at 0x1c3b4e140>
Other <cell2mol.tmcharge_common.molecule object at 0x1c3b4fe50>
7 Species (Ligand or Molecules) to Characterize


In [26]:
newcell.speclist

In [27]:
selected_charge_states, Warning = drive_get_poscharges(unique_species, debug=debug)


    ---------------
    #### Ligand ####
    ---------------
           1 Cl 1
Charge state and protonation received for molecule 1

    ---------------
    #### Ligand ####
    ---------------
           1 Cl 2
Charge state and protonation received for molecule 1

    ---------------
    #### Metal ####
    ---------------
           Fe coordination_sphere	 ['Cl', 'Cl', 'Cl', 'Cl']
           Fe coordinating_atoms	 ['Cl', 'Cl', 'Cl', 'Cl']
Possible charges received for metal: [2, 3]

    ---------------
    #### NON-Complex ####
    ---------------
           15 H10-C4-O
Charge state and protonation received for molecule 1

    ---------------
    #### Ligand ####
    ---------------
           3 H2-O 1
Charge state and protonation received for molecule 1

    ---------------
    #### Metal ####
    ---------------
           Fe coordination_sphere	 ['Cl', 'Cl', 'Cl', 'O']
           Fe coordinating_atoms	 ['Cl', 'Cl', 'Cl', 'O']
Possible charges received for metal: [2, 3]

    -----

In [36]:
selected_charge_states

[[[<cell2mol.formal_charge.charge_state at 0x1c3cbae30>,
 [[<cell2mol.formal_charge.charge_state at 0x1c3cb9d50>,
 [],
 [[<cell2mol.formal_charge.charge_state at 0x1c3cb8370>,
 [[<cell2mol.formal_charge.charge_state at 0x1c3cb83d0>,
 [],
 [[<cell2mol.formal_charge.charge_state at 0x1c3cb9720>,

In [45]:
for idx, sel in enumerate(selected_charge_states):
    for jdx, opt in enumerate(sel):
        chstate = opt[0]
        prot = opt[1]
        if debug >= 1: print(f"PREPARE: State {idx} and option {jdx}. Target state and protonation received with {chstate.corr_total_charge} and {prot.added_atoms}")
        print(chstate.smiles)
        print(chstate.addedlist, chstate.metal_electrons, chstate.elemlist)    
        print(prot.addedlist, prot.metal_electrons, prot.elemlist)

PREPARE: State 0 and option 0. Target state and protonation received with -1 and 1
[H]Cl
[1] [0] ['H']
[1] [0] ['H']
PREPARE: State 1 and option 0. Target state and protonation received with -1 and 1
[H]Cl
[1] [0] ['H']
[1] [0] ['H']
PREPARE: State 3 and option 0. Target state and protonation received with 0 and 0
[H]C([H])([H])C([H])([H])OC([H])([H])C([H])([H])[H]
[] [] []
[] [] []
PREPARE: State 4 and option 0. Target state and protonation received with 0 and 0
[H]O[H]
[0 0 0] [0 0 0] ['0.0' '0.0' '0.0']
[0 0 0] [0 0 0] ['0.0' '0.0' '0.0']
PREPARE: State 6 and option 0. Target state and protonation received with 1 and 0
[H]C([H])([H])C([H])([H])C([H])([H])C([H])([H])[N+](C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])(C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]
[] [] []
[] [] []


In [29]:
final_charge_distribution = balance_charge(unique_indices,unique_species,debug=debug)

In [30]:
final_charge_distribution

[[-1,
  -1,
  -1,
  -1,
  -1,
  -1,
  2,
  2,
  0,
  0,
  -1,
  -1,
  -1,
  2,
  0,
  0,
  -1,
  -1,
  -1,
  2,
  1,
  1,
  1,
  1]]

In [32]:
prepare_mols(newcell.moleclist, 
             unique_indices, 
             unique_species, 
             selected_charge_states, 
             final_charge_distribution[0], 
             debug=2)

PREPARE: 7 selected_charge_states received

PREPARE: 9 molecules to prepare, of types
PREPARE: Molecule 0 is a Complex with formula Cl6-Fe2
PREPARE: Molecule 1 is a Other with formula H10-C4-O
PREPARE: Molecule 2 is a Complex with formula H2-O-Cl3-Fe
PREPARE: Molecule 3 is a Other with formula H10-C4-O
PREPARE: Molecule 4 is a Complex with formula H2-O-Cl3-Fe
PREPARE: Molecule 5 is a Other with formula H36-C16-N
PREPARE: Molecule 6 is a Other with formula H36-C16-N
PREPARE: Molecule 7 is a Other with formula H36-C16-N
PREPARE: Molecule 8 is a Other with formula H36-C16-N

PREPARE: State 0 and option 0. Target state and protonation received with -1 and 1
PREPARE: State 1 and option 0. Target state and protonation received with -1 and 1
PREPARE: State 3 and option 0. Target state and protonation received with 0 and 0
PREPARE: State 4 and option 0. Target state and protonation received with 0 and 0
PREPARE: State 6 and option 0. Target state and protonation received with 1 and 0

PREPARE:

([<cell2mol.tmcharge_common.molecule at 0x1c38e9f00>,
 False)